In [1]:
!pip install torch transformers datasets

In [2]:
!pip install -U transformers accelerate datasets torch
## restart kernel after running 

   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ------- -------------------------------- 2.1/10.8 MB 10.4 MB/s eta 0:00:01
   --------------- ------------------------ 4.2/10.8 MB 10.2 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.8 MB 10.4 MB/s eta 0:00:01
   ---------------------------------- ----- 9.2/10.8 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 9.9 MB/s  0:00:01
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 529.0/529.0 kB 7.5 MB/s  0:00:00

  Attempting uninstall: datasets

    Found existing installation: datasets 4.8.4

   ---------------------------------------- 0/2 [datasets]
    Uninstalling datasets-4.8.4:
   ---------------------------------------- 0/2 [datasets]
      Successfully uninstalled datasets-4.8.4
   ---------------------------------------- 0/2 [datasets]
   ---------------------------------------- 0/2 [datase

In [3]:
import tkinter as tk
import transformers
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification , TrainingArguments, Trainer                          

In [4]:
from datasets import load_dataset

In [5]:
# ---------------- LOAD SMALL DATA ----------------
dataset = load_dataset("go_emotions")

# 🔥 TAKE SMALL SAMPLE (VERY IMPORTANT)
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(5000))

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ---------------- TOKENIZE ----------------
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length")

dataset = dataset.map(tokenize, batched=True)

# ---------------- FIX LABELS ----------------
def convert_labels(example):
    if len(example["labels"]) > 0:
        example["label"] = example["labels"][0]
    else:
        example["label"] = 27
    return example

dataset = dataset.map(convert_labels)
dataset = dataset.remove_columns(["labels"])

# ---------------- MODEL ----------------
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=28
)

# ---------------- TRAINING ----------------
training_args = TrainingArguments(
    output_dir="./emotion_model",
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=200,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"]
)

trainer.train()

# ---------------- SAVE ----------------
model.save_pretrained("emotion_detection1")
tokenizer.save_pretrained("emotion_detection1")

print("✅ FAST TRAINING COMPLETE")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAST TRAINING COMPLETE
